In [1]:
import numpy as np
import pandas as pd
import glob
import os
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.io import ascii
from astropy.table import Table,join
import itertools
from astroquery.simbad import Simbad
from utils import PLOT_PARAMS
PLOT_PARAMS()
from datetime import date
from astropy.coordinates import SkyCoord, EarthLocation

today=date.today()
DATE =today.strftime("%m_%d_%y")
print(DATE)
import pickle
import thejoker as tj
import pymc as pm
import corner
import arviz as az
plt.rcParams['figure.facecolor'] = 'white'
import warnings
warnings.filterwarnings("ignore")

11_17_26


In [2]:
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
import astropy.units as u

vlt = EarthLocation.of_site('paranal')  # the easiest way... but requires internet

In [3]:
files_p112 = glob.glob('../data/P112_data/*.fits')
files_p113 = glob.glob('../data/P113_data/*.fits')
files_p116 = glob.glob('../data/P116_data/*.fits')


In [4]:
def get_galah():
    galah =(Table.read('../../lithium/data/GALAH_DR3_main_allstar_v2.fits',format='fits'))#.to_pandas()

    dr2_ids = np.array(galah['dr2_source_id']).astype('str')
    dr3_ids = np.array(galah['dr3_source_id']).astype('str')
    
    galah.remove_column('dr2_source_id')
    galah.remove_column('dr3_source_id')
    galah = galah.to_pandas()

    galah.insert(2, 'dr2_source_id', dr2_ids)
    galah.insert(3, 'dr3_source_id', dr3_ids)
    
    galah['ph_qual_wise']=galah['ph_qual_wise'].str.decode("utf-8")
    return galah
galah=get_galah()

In [5]:
galah_rv = Table.read('/Users/maryumsayeed/Desktop/Code/Research/lithium/data/GALAH_DR3_VAC_rv_v2.fits').to_pandas()

In [6]:
def get_dir(files):
    obj_dir = {}

    counts = []
    for i,file in enumerate(files):
        with fits.open(file) as hdul:
            hdul = fits.open(file)
            header  = hdul[0].header
            #print(header.cards['HIERARCH ESO QC CCF RV ERROR'])
            data    = hdul[1].data
            time    = header['HIERARCH ESO QC BJD']
            name    = header['OBJECT']
            obj     = int(header['HIERARCH ESO OBS NAME'].split('-')[0][2:])
            snr     = header['SNR']
            counts.append(obj)

            grade   = header['OB_GRADE'].replace(" ", "")  
            if grade =='C':
                print('RG',obj,header['OB_GRADE'])
                continue

            ra,dec  = header['RA'], header['DEC']
            obs_T   = header['DATE-OBS'].split('T')[0]
            rv_val  =  header['HIERARCH ESO QC CCF RV']#header['HIERARCH ESO OCS OBJ RV'] #km/s
            rv_err  = header['HIERARCH ESO QC CCF RV ERROR'] #km/s
            sc = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)
            rv_corr = rv_val
            err_corr = rv_err

            if obj in obj_dir.keys():
                obj_dir[obj]['rv'].append(rv_corr)
                obj_dir[obj]['time'].append(time)
                obj_dir[obj]['err'].append(err_corr)
                obj_dir[obj]['name'].append(name)
                obj_dir[obj]['grade'].append(grade)
                obj_dir[obj]['snr'].append(snr)
                
            else:
                obj_dir[obj] = {'rv': [rv_corr], 'time': [time], 'err': [err_corr], 
                                'name': [name], 'grade': [grade], 'snr': [snr]}
          
    targets_to_remove = []
    for k, v in obj_dir.items():
        if len(v['time']) < 3:
            targets_to_remove.append(k)
    for i in range(len(targets_to_remove)):
        del obj_dir[targets_to_remove[i]]
    print(len(obj_dir.keys()))

    myKeys = list(obj_dir.keys())
    myKeys.sort()
    obj_dir = {i: obj_dir[i] for i in myKeys}

    return obj_dir

obj_dir_p112 = get_dir(files_p112)
# print(len(obj_dir_p112))
del obj_dir_p112[13] #13 ended up being a dipper star
obj_dir_p113 = get_dir(files_p113)

obj_dir_p116 = get_dir(files_p116)

RG 20 C
RG 13 C
RG 14 C
RG 1 C
10
RG 12 C
RG 6 C
RG 3 C
RG 11 C
RG 22 C
RG 15 C
RG 3 C
RG 8 C
RG 3 C
RG 25 C
RG 8 C
RG 22 C
RG 3 C
RG 7 C
RG 23 C
25
RG 16 C
RG 21 C
RG 13 C
25


In [8]:
def combine_table(obj_dir,program):
    new_df = {'RG_id':[], 'galah_id': [], 'obj_name': [], 'esp_time':[],'esp_rv':[],'esp_rv_err':[],
              'program':[]
             }

    for k, v in obj_dir.items():
        if len(v['rv']) < 3: 
            continue
        result_table = Simbad.query_objectids(v['name'][0])
        result_table = np.array(result_table).astype('str')

        for i in result_table:
            if 'GALAH' in i: 
                galah_name = i.split(' ')[-1]
        if v['name'][0] == 'UCAC4 202-005918':
            galah_name  = 171208003602294
                
        new_df['RG_id'].append(k)
        new_df['program'].append(program)
        new_df['galah_id'].append(int(galah_name))
        new_df['obj_name'].append(v['name'][0])
        new_df['esp_rv'].append(v['rv'])
        new_df['esp_time'].append(v['time'])
        new_df['esp_rv_err'].append(v['err'])
        
    new_df = pd.DataFrame(new_df)

    cols = ['sobject_id', 'vbroad', 'fe_h', 'e_fe_h' ,'Li_fe', 'e_Li_fe','ruwe_dr2','e_vbroad', 'Ba_fe', 'e_Ba_fe','La_fe','e_La_fe','Zr_fe', 'e_Zr_fe','Y_fe','e_Y_fe']
    new_df = pd.merge(new_df, galah[cols], left_on='galah_id', right_on='sobject_id')
    new_df['Li_val'] = new_df.fe_h+new_df.Li_fe+1.05
    new_df['e_Li_val'] = np.sqrt(new_df.e_fe_h**2.+new_df.e_Li_fe**2.)

    cols = ['RG_id', 'galah_id', 'obj_name', 
            'esp_time','esp_rv','esp_rv_err',
            'Li_val', 'e_Li_val','vbroad','e_vbroad', 'fe_h','ruwe_dr2', 
            'Ba_fe', 'e_Ba_fe','La_fe','e_La_fe','Zr_fe', 'e_Zr_fe','Y_fe','e_Y_fe',
            'program'
           ]
    new_df = new_df[cols]
    new_df.sort_values(by=['RG_id'],inplace=True)
    
    new_df = new_df.round(2)
    return new_df

df_p112 = combine_table(obj_dir_p112,program=112)
df_p113 = combine_table(obj_dir_p113,program=113)
df_p116 = combine_table(obj_dir_p116,program=116)


DALServiceError: Unable to access the capabilities endpoint at:
- https://simbad.cds.unistra.fr/simbad/sim-tap/capabilities: 404 Not Found
- https://simbad.cds.unistra.fr/simbad/capabilities: 404 Not Found

This could mean:
1. The service URL is incorrect
2. The service is temporarily unavailable
3. The service doesn't support this protocol
4. If a 503 was encountered, retry after the suggested delay.


In [ ]:
p112_galah = pd.merge(galah[['sobject_id','logg','teff']], df_p112,left_on='sobject_id', right_on='galah_id', how='right')
p113_galah = pd.merge(galah[['sobject_id','logg','teff']], df_p113,left_on='sobject_id', right_on='galah_id', how='right')
p116_galah = pd.merge(galah[['sobject_id','logg','teff']], df_p116,left_on='sobject_id', right_on='galah_id', how='right')


In [ ]:
for i in p116_galah['sobject_id']:
    print(i)

In [ ]:
# one target had data taken in P112 and P116. UCAC4 202-005918 or DR3_id = 171208003602294
# combine the espresso measurements for this target
p112_galah[p112_galah['sobject_id']==171208003602294]#['RG_id']

In [ ]:
# combine the espresso measurements between P112 and P116:

c='esp_time'
merge_time = np.concatenate([p112_galah[p112_galah['sobject_id']==171208003602294][c].values, p116_galah[p116_galah['sobject_id']==171208003602294][c].values])
merge_time = list(itertools.chain(*merge_time))

c='esp_rv'
merge_rv = np.concatenate([p112_galah[p112_galah['sobject_id']==171208003602294][c].values, p116_galah[p116_galah['sobject_id']==171208003602294][c].values])
merge_rv = list(itertools.chain(*merge_rv))

c='esp_rv_err'
merge_rv_err = np.concatenate([p112_galah[p112_galah['sobject_id']==171208003602294][c].values, p116_galah[p116_galah['sobject_id']==171208003602294][c].values])
merge_rv_err = list(itertools.chain(*merge_rv_err))

print(len(merge_time))
print(len(merge_rv))
print(len(merge_rv_err))

# save the combined list in the P116 table

new_p116_galah = p116_galah.copy()

index = new_p116_galah.loc[new_p116_galah['sobject_id']==171208003602294].index[0]
new_p116_galah.at[index,'esp_time']   = merge_time
new_p116_galah.at[index,'esp_rv']     = merge_rv
new_p116_galah.at[index,'esp_rv_err'] = merge_rv_err

# Remove the target from P112 table that was also observed in P116:
new_p112_galah = p112_galah[p112_galah['sobject_id']!=171208003602294]


In [ ]:


all_112_err = [item for sublist in new_p112_galah['esp_rv_err'].values for item in sublist]
all_113_err = [item for sublist in p113_galah['esp_rv_err'].values for item in sublist]
all_116_err = [item for sublist in new_p116_galah['esp_rv_err'].values for item in sublist]


plt.hist(all_112_err,bins='auto',density=True,label='112',histtype='step',color='k',lw=2)
plt.hist(all_113_err,bins='auto',density=True,label='113',histtype='step',color='r',lw=2)
plt.hist(all_116_err,bins='auto',density=True,alpha=0.3,label='116',color='grey')
plt.legend()

plt.axvline(np.median(all_112_err),c='k',lw=2,ls='dashed')
plt.axvline(np.median(all_113_err),c='r',lw=2,ls='dashed')
plt.axvline(np.median(all_116_err),c='grey',lw=2,ls='dashed')

print('112: \t %.3f +/- %.3f'%(np.median(all_112_err),np.std(all_112_err)/np.sqrt(len(all_112_err))))
print('113: \t %.3f +/- %.3f'%(np.median(all_113_err),np.std(all_113_err)/np.sqrt(len(all_113_err))))
print('116: \t %.3f +/- %.3f'%(np.median(all_116_err),np.std(all_116_err)/np.sqrt(len(all_116_err))))

In [ ]:
print(len(new_p112_galah))
print(len(p113_galah))
print(len(new_p116_galah))

df_both = pd.concat([new_p112_galah, p113_galah, new_p116_galah])
df_both = pd.merge(df_both, galah[['star_id','sobject_id','ra_dr2','dec_dr2','dr3_source_id']],on='sobject_id')
df_both = pd.merge(df_both, galah_rv[['sobject_id','MJD_local','rv_obst','rv_sme_v2','rv_nogr_obst','e_rv_nogr_obst','rv_galah','e_rv_galah','dr2_radial_velocity','use_rv_flag']],on='sobject_id')

print('total targets:',len(df_both))

In [ ]:
def convert_mjd_to_jd(mjd):
    t = Time(mjd, format='mjd', scale='local')
    return t.jd

# for i, row in df_both.iterrows():
#     convert_mjd_to_jd(row['MJD_local'])
convert_mjd_to_jd(df_both[df_both['MJD_local']==df_both['MJD_local']]['MJD_local'])

In [ ]:
def comb_data(df):
    new_df = {'RG_id':[],'sobject_id':[],
              'esp_time': [], 'esp_rv': [], 'esp_rv_err': [],
             }
#               'comb_time':[],'comb_rv':[],'comb_rv_err':[]}

    for i, row in df.iterrows():
        if str(row['MJD_local']) == 'nan':
            galah_bmjd = 'nan'
        else:
            galah_bmjd = convert_mjd_to_jd(row['MJD_local'])
        t = row['esp_time']*1
        r = row['esp_rv']*1
        e = row['esp_rv_err']*1
        
        t, r, e = np.array(t), np.array(r), np.array(e)

        if row.RG_id!=5:
            mask    = (r >= -100)
            t, r, e = t[mask], r[mask], e[mask]
        t, r, e = list(t), list(r), list(e)
        new_df['esp_time'].append(t)
        new_df['esp_rv'].append(r)
        new_df['esp_rv_err'].append(e)

#         t.append(galah_bmjd)
#         r.append(row['rv_galah'])
#         e.append(row['e_rv_galah'])
        
#         mask = [i for i in range(len(t)) if t[i] != 'nan']
#         t, r, e = np.array(t),np.array(r),np.array(e)

        new_df['RG_id'].append(row.RG_id)
        new_df['sobject_id'].append(row.sobject_id)
#         new_df['comb_time'].append(t[mask])
#         new_df['comb_rv'].append(r[mask])
#         new_df['comb_rv_err'].append(e[mask])
        
    new_df = pd.DataFrame(new_df)
    return new_df

In [ ]:
def clean_up_rvs(df):
    
    new_df = {'sobject_id':[],'esp_time':[],'esp_rv':[],'esp_rv_err':[]}
    
    for i, row in df.iterrows():
        rv_vals = np.array(row.esp_rv)
        yerr    = np.array(row.esp_rv_err)
        time    = np.array(row.esp_time)
        if row.RG_id!=5:
            mask    = (rv_vals >= -100)
            time, rv_vals, yerr = time[mask], rv_vals[mask], yerr[mask]
            
        new_df['sobject_id'].append(row.sobject_id)
        new_df['esp_time'].append(time)
        new_df['esp_rv'].append(rv_vals)
        new_df['esp_rv_err'].append(yerr)
        if row.program==113 and row.RG_id==14:
            print(rv_vals)
    new_df = pd.DataFrame(new_df)
    return new_df


In [ ]:
# new_df = comb_data(df_both)
new_df = clean_up_rvs(df_both)

In [ ]:

df_both_1 = df_both.drop(columns=['esp_time','esp_rv','esp_rv_err'])
comb_df = pd.merge(df_both_1, new_df, on='sobject_id')
comb_df.head(1)


In [ ]:
def get_rv_variation(df):
    new_df = df.copy()
    all_diff =[]
    for i, row in new_df.iterrows():
        diff = abs(max(row['esp_rv'])-min(row['esp_rv']) )
        all_diff.append(diff)
    
    
    new_df['rv_var'] = all_diff
    #new_df[['sobject_id','rv_var']].to_csv('rv_var_df.csv',index=False)
    #p113[['sobject_id','rv_var']].to_csv('p113_rv_var.csv',index=False)
    #p116[['sobject_id','rv_var']].to_csv('p116_rv_var.csv',index=False)
    
    
get_rv_variation(new_df)


In [ ]:
# comb_df[['ra','dec']]
comb_df.keys()

aa = pd.merge(galah[['sobject_id','dr3_source_id','ra_dr2','dec_dr2']],
         comb_df['sobject_id'],left_on='sobject_id',right_on='sobject_id').to_csv('need_for_rave.csv',index=False)


In [ ]:
# comb_df.to_csv('rv_df_new.csv',index=False)


In [ ]:
all_dfs = []

for i, row in comb_df.iterrows():
    N      = np.array(row.esp_time)
    ids    = [row.sobject_id]*len(N)
    source = ['ESPRESSO']*len(N)
    
    dat1   = pd.DataFrame({'GALAH_ID':ids,
                        'time':np.array(row.esp_time).astype(str), 
                        'rv':np.array(row.esp_rv).astype(str),  
                        'rv_err':np.array(row.esp_rv_err).astype(str),
                        'source':source
                       })

    N      = np.array([row.rv_galah])
    ids    = [row.sobject_id]*len(N)
    source = ['GALAH']*len(N)
    time = row.MJD_local

    if time != time:
        new_time = np.nan
    else:
        new_time = convert_mjd_to_jd(time)

    
    dat2   = pd.DataFrame({'GALAH_ID':ids,
                        'time':np.array(new_time).astype(str),   
                        'rv':np.array(row.rv_galah).astype(str),  
                        'rv_err':np.array(row.e_rv_galah).astype(str),
                        'source':source
                       })
    
    dat = pd.concat([dat1, dat2])
    all_dfs.append(dat)


In [ ]:
#pd.concat(all_dfs).to_csv('rv_table.csv',index=False)

In [ ]:
comb_df_tb = Table.from_pandas(comb_df)
comb_df_tb['dr3_source_id'] = comb_df_tb['dr3_source_id'].astype(int)

In [ ]:
def convert_time_to_jd(time,date_format=False):
    from datetime import datetime
    from astropy.time import Time
    
    if date_format:
        # Create a datetime object
        dt = datetime.strptime(time, date_format)
    else:
        # Create a datetime object
        dt = datetime.strptime(time, "%Y%m%d")

    # Convert the datetime object to an astropy Time object
    t = Time(dt, format='datetime', scale='utc')
    
    # Access the Modified Julian Date (MJD)
    jd = t.jd
    
    return jd

# Add RAVE data

In [ ]:
#rave = Table.read('/Users/maryumsayeed/Downloads/rave_crossmatch.fits')
rave = Table.read('../files/rave_crossmatch.fits')

In [ ]:
rave

In [ ]:
joined = join(rave,comb_df_tb,keys_left='sobject_id',keys_right='sobject_id')#
joined['rave_obsdate'] = [i.split('_')[0] for i in joined['ObsID']]
joined['rave_jd']     = [convert_time_to_jd(i,date_format="%Y-%m-%d") for i in joined['Obs.date']]
joined['source']       = ['RAVE']*len(joined)

rave_joined = pd.DataFrame({'GALAH_ID':joined['sobject_id_1'] ,
                            'time':joined['rave_jd'] ,
                            'rv':joined['HRV'] ,
                            'rv_err':joined['e_HRV'] ,
                            'source':joined['source'] ,
                           })

esp_galah = pd.concat(all_dfs)
esp_galah_rave = pd.concat([esp_galah,rave_joined])


In [ ]:
esp_galah_rave[esp_galah_rave.source=='RAVE']

In [ ]:
len(esp_galah_rave['GALAH_ID'].unique())

# Add APOGEE DR17 Data

In [ ]:
adr17=pd.read_csv('../data/apogee_dr17.tsv',delimiter=';')
adr17['APOGEE'] = adr17['APOGEE'].astype(str)
adr17['APOGEE'] = [i.strip() for i in adr17['APOGEE']]


In [ ]:
# Find 2MASS ID on Simbad. Remove the J in front of the ID. Enter "2M" then this ID in APOGEE DR17 catalog.

In [ ]:
adr17_galah_df = df_both['star_id'].copy()
get_sample_2mass_id = [str(i).replace("'","") for i in df_both['star_id']]
get_sample_2mass_id = [str(i).replace("b","") for i in get_sample_2mass_id]
get_sample_2mass_id = ['2M'+i for i in get_sample_2mass_id]
adr17_galah_df = pd.DataFrame({'sobject_id':df_both['sobject_id'], 
                               'star_id':df_both['star_id'],
                               '2MASS':get_sample_2mass_id})
# adr17_galah_df

In [ ]:
overlap_adr17_and_sample = pd.merge(adr17, adr17_galah_df, left_on='APOGEE',right_on='2MASS')
overlap_adr17_and_sample[['sobject_id','star_id','APOGEE','JD','RV','VHelio','e_RV','BC','Chi2RV']]

# Use VHelio in APOGEE. This is barycentric correction and NOT heliocentric but the name is kept for 
# historical reasons. See this link at this link: https://www.sdss4.org/dr17/irspec/radialvelocities/
# "Note: The terminology and header keyword VHELIO is retained for historical reasons, 
# but the pipeline is calculating a true barycentric velocity, not a heliocentric velocity."

In [ ]:
overlap_adr17_and_sample['source'] = ['APOGEE']*len(overlap_adr17_and_sample)
overlap_adr17_and_sample.rename(columns={'JD':'time',
                                         'VHelio':'rv',
                                         'e_RV':'rv_err',
                                         'sobject_id':'GALAH_ID'},inplace=True)

# apogee_dr17 = {
#     'GALAH_ID' : [150428000601308,151230003202196,140707003101315,140707003101315],
#     'time' : [2458646.67558,2458827.64986,2458732.79792,2458731.78777],
#     'rv': [27.24648666381836,33.2837028503418,-44.76622461216243,-44.57581773079781],
#     'rv_err':[0.027553709223866463,0.029930321499705315,0.05547336542377407,0.058634062254277884],
#     'source': ['APOGEE', 'APOGEE','APOGEE','APOGEE']
# }
overlap_adr17_and_sample_short = overlap_adr17_and_sample[list(esp_galah_rave.keys())]

esp_galah_rave_apogee = pd.concat([esp_galah_rave, overlap_adr17_and_sample_short])

esp_galah_rave_apogee = esp_galah_rave_apogee.sort_values(by=['GALAH_ID','source'])

esp_galah_rave_apogee



In [ ]:
esp_galah_rave[esp_galah_rave.source=='RAVE']

In [ ]:
esp_galah_rave_apogee[esp_galah_rave_apogee.source=='RAVE']

In [ ]:
# Get stats:
e = 'ESPRESSO'
a = 'APOGEE'
g = 'GALAH'
r = 'RAVE'

e_only = esp_galah_rave_apogee[esp_galah_rave_apogee['source'].isin([e])]['GALAH_ID'].unique()
g_only = esp_galah_rave_apogee[esp_galah_rave_apogee['source'].isin([g])]['GALAH_ID'].unique()
a_only = esp_galah_rave_apogee[esp_galah_rave_apogee['source'].isin([a])]['GALAH_ID'].unique()
r_only = esp_galah_rave_apogee[esp_galah_rave_apogee['source'].isin([r])]['GALAH_ID'].unique()

print('Targets with only ESPRESSO data:',len(set(e_only)))
print('Targets with ESPRESSO & GALAH data:',len(set(e_only) & set(g_only)))
print('Targets with ESPRESSO, GALAH & RAVE data:',len(set(e_only) & set(g_only) & set(r_only)))
print('Targets with ESPRESSO, GALAH & APOGEE data:',len(set(e_only) & set(g_only) & set(a_only)))
print('Targets with ESPRESSO, GALAH, RAVE & APOGEE data:',len(set(e_only) & set(g_only) & set(r_only) & set(a_only)))


In [ ]:
#esp_galah_rave_apogee.to_csv('combined_rvs_new.csv',index=False)

In [ ]:
ss

In [ ]:
paper_table = esp_galah_rave_apogee.copy()
for c in ['time','rv','rv_err']:    
    paper_table[c] = paper_table[c].astype(float)


paper_table['time'] = paper_table['time'].round(1)
paper_table['rv'] = paper_table['rv'].round(1)
paper_table['rv_err'] = 1000*paper_table['rv_err']#.round(1)

paper_table['rv_err'] = ['%i' % i for i in paper_table['rv_err']]
print(paper_table.head(20).to_latex(index=False,float_format="{:.2f}".format))

paper_table.keys()

In [ ]:
ss

In [18]:
#paper_table.to_csv('../submission_revised_1/Table2.csv',index=False)


def _():
    a = pd.read_csv('../submission_AAS_2/Table2.csv')
    a = a.round({"rv": 4,
                 "rv_err": 4,
                 "time":4})
    
    a['GALAH_ID'] = a['GALAH_ID'].astype('str')
    a['source']   = a['source'].astype('str')
    a.to_csv('Table2_reformat.csv',index=False)
    return a
_()

,GALAH_ID,time,rv,rv_err,source
0,140209002201006,2.461084e+06,159.8670,0.0718,ESPRESSO
1,140209002201006,2.461001e+06,159.8317,0.0673,ESPRESSO
2,140209002201006,2.461099e+06,159.6528,0.0662,ESPRESSO
3,140209002201006,2.461068e+06,159.9275,0.0651,ESPRESSO
4,140209002201006,2.461039e+06,159.7899,0.0737,ESPRESSO
...,...,...,...,...,...
367,181226004101033,2.461014e+06,76.5601,0.0247,ESPRESSO
368,181226004101033,2.461114e+06,76.5789,0.0203,ESPRESSO
369,181226004101033,2.461068e+06,76.6091,0.0248,ESPRESSO
370,181226004101033,2.461039e+06,76.6016,0.0276,ESPRESSO


In [ ]:
paper_table[paper_table.source=='RAVE']

In [ ]:
ss

In [ ]:
ss

In [ ]:
comb_df[comb_df.sobject_id==150428000601308]['program']

In [ ]:
ss

In [ ]:
def make_plot(df):
    plt.figure(figsize=(8,30))
    for i, row in df.iterrows():
        rv_vals = np.array(row.comb_rv)
        yerr    = np.array(row.comb_rv_err)
        time    = np.array(row.comb_time)

        t2 = Time(time, format='jd')

        bmjd2 = t2.tcb.mjd

        bmjd2 = bmjd2 - min(bmjd2)
        plt.subplot(17,2,i+1)
        plt.scatter(bmjd2,rv_vals)

        peak = max(rv_vals) - min(rv_vals)
        plt.title('RG%s (%s) - %.1f'%(row.RG_id, row.program, peak),fontsize=8)
    plt.tight_layout()
    plt.show()
make_plot(comb_df)

In [ ]:
cols = ['sobject_id', 'dr3_source_id','ra_dr2', 'dec_dr2','obj_name','logg', 'teff',  'Li_val', 'vbroad', 'fe_h', 
        'ruwe_dr2']
# both[cols].to_csv('espresso_targets.csv',index=False)